In [12]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader

In [13]:
# Lấy thư mục hiện tại nơi đang chạy notebook
CURRENT_DIR = os.getcwd()

# Kiểm tra thông minh: Nếu notebook đang nằm trong thư mục 'src' thì lùi ra 1 cấp,
# còn nếu đã nằm ở ngoài cùng thì giữ nguyên.
if os.path.basename(CURRENT_DIR) == "src":
    PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, ".."))
else:
    PROJECT_ROOT = CURRENT_DIR

# Trỏ chính xác tới 2 thư mục Data và model nằm ở ngoài
DATA_DIR = os.path.join(PROJECT_ROOT, "Data")
MODEL_DIR = os.path.join(PROJECT_ROOT, "model")

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {device}")

Đang sử dụng thiết bị: cpu


In [15]:
class TripletCIFAR10(Dataset):
    def __init__(self, cifar_dataset):
        self.cifar_dataset = cifar_dataset
        # Lấy danh sách nhãn
        self.labels = np.array(cifar_dataset.targets)
        # Gom index của các ảnh theo từng class (0 đến 9)
        self.class_indices = [np.where(self.labels == i)[0] for i in range(10)]

    def __len__(self):
        return len(self.cifar_dataset)

    def __getitem__(self, idx):
        # 1. Anchor (Ảnh gốc)
        anchor_img, label = self.cifar_dataset[idx]

        # 2. Positive (Ảnh cùng class với Anchor)
        positive_idx = idx
        while positive_idx == idx:  # Đảm bảo không bốc trùng ảnh Anchor
            positive_idx = np.random.choice(self.class_indices[label])
        positive_img, _ = self.cifar_dataset[positive_idx]

        # 3. Negative (Ảnh khác class với Anchor)
        negative_label = np.random.choice([i for i in range(10) if i != label])
        negative_idx = np.random.choice(self.class_indices[negative_label])
        negative_img, _ = self.cifar_dataset[negative_idx]

        return anchor_img, positive_img, negative_img

In [16]:
# Tiền xử lý ảnh
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [17]:
# Tải dữ liệu CIFAR-10 gốc
base_trainset = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=transform)

# Bọc CIFAR-10 vào Triplet Dataset
triplet_trainset = TripletCIFAR10(base_trainset)
train_loader = DataLoader(triplet_trainset, batch_size=64, shuffle=True, num_workers=0)

print(f"Đã chuẩn bị xong dữ liệu Triplet! Số lượng mẫu: {len(triplet_trainset)}")

Files already downloaded and verified
Đã chuẩn bị xong dữ liệu Triplet! Số lượng mẫu: 50000


In [18]:
# 3. KHỞI TẠO MÔ HÌNH BỎ LỚP FC VÀ HÀM LOSS
# ==============================================================================
# Load ResNet-18 pre-trained
model = models.resnet18(weights="IMAGENET1K_V1")

In [19]:
# CẮT BỎ LỚP FC: Thay lớp cuối cùng bằng nn.Identity() (Không làm gì cả)
# Ảnh đi qua mạng sẽ giữ nguyên dạng vector 512 chiều.
model.fc = nn.Identity()  
model = model.to(device)

In [20]:
# Hàm Triplet Loss: Cố gắng ép khoảng cách (Positive - Negative) >= 1.0 (margin)
criterion = nn.TripletMarginLoss(margin=1.0, p=2)

# Dùng Learning Rate RẤT NHỎ (1e-4) để không phá hỏng các đặc trưng màu sắc có sẵn
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [21]:
# 4. VÒNG LẶP HUẤN LUYỆN (TRAINING LOOP)
# ==============================================================================
num_epochs = 5  # Train khoảng 5-10 epoch là thấy sự khác biệt

print("\n>>> BẮT ĐẦU HUẤN LUYỆN VỚI TRIPLET LOSS...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for batch_idx, (anchor, positive, negative) in enumerate(train_loader):
        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
        
        optimizer.zero_grad()
        
        # Đưa cả 3 ảnh qua mạng để lấy 3 vector 512 chiều
        vector_anchor = model(anchor)
        vector_positive = model(positive)
        vector_negative = model(negative)
        
        # Tính khoảng cách và Loss
        loss = criterion(vector_anchor, vector_positive, vector_negative)
        
        # Cập nhật trọng số
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        if (batch_idx + 1) % 100 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] | Batch [{batch_idx+1}/{len(train_loader)}] | Loss: {loss.item():.4f}")
            
    epoch_loss = running_loss / len(train_loader)
    print(f"==> KẾT THÚC EPOCH {epoch+1} | Average Loss: {epoch_loss:.4f}\n")


>>> BẮT ĐẦU HUẤN LUYỆN VỚI TRIPLET LOSS...


KeyboardInterrupt: 